# Finite-ground CPW — Q2D cross section

Extract per-unit-length matrices from an independently authored CPW
cross section. See the [Q2D
requirements](../../../../simulation/requirements/q2d-cross-section.html).
This special model retains a 10 µm signal, 10 µm gaps, 40 µm grounds and
Nb 200 nm above a 200 µm wide, 500 µm deep Si substrate. It is not a 3D
coupon slice or the current-PDK Al model. Native rectangles and
conductor assignments define the case; no GDS is imported.

Fresh construction uses the local shared-physical candidate, not a
published wheel. Retained results keep their original runtime
identities.

Published historical baseline:
[SCGSim](https://github.com/OrPenStrike/scgsim/tree/c9270c7e76e0b2df187dddde462079e00d410dd0).

## Setup and Imports

In [ ]:
from __future__ import annotations

import subprocess
from pathlib import Path
from uuid import uuid4

from IPython.display import display
from scgsim.aedt import (
    AedtResources,
    MatrixRunControl,
    PdkMaterial,
    Q2dConductorSpec,
    Q2dRectangleSpec,
    Q2dSpec,
    prepare_handoff,
    resolve_results,
)

from orcapen_sc_pdk.materials import get_material_records

## Setup and Run Controls

In [ ]:
WORKFLOW_ACTION = "run"  # AUTHORING_ONLY | prepare_handoff | run | analyze_handoff
RUN_ID = f"cpw_q2d_{uuid4().hex}"
OUTPUT_ROOT = Path.cwd() / ".artifacts" / "SCGSimExamples" / "CpwFiniteGround"
RUN_DIR = OUTPUT_ROOT / RUN_ID
RETURNED_RUN_DIR = None  # Set independently to an immutable returned-run directory.
SCGSIM_RUNTIME_SOURCE = "LOCAL_SHARED_PHYSICAL_CANDIDATE_UNCOMMITTED"
RESOURCES = AedtResources(cores=4, ram_limit_percent=30)
# Cross-section signal width (um).
signal_width_um = 10.0
# Signal-to-ground gap (um).
gap_um = 10.0
# Ground conductor width (um).
ground_width_um = 40.0
# Substrate width (um).
substrate_width_um = 200.0
# Metal thickness (um).
metal_thickness_um = 0.2
# Substrate thickness (um).
substrate_thickness_um = 500.0
# AEDT version used for project generation.
aedt_version = "2024.2"
# AEDT project name.
project_name = RUN_ID
# AEDT design name.
design_name = "CpwFiniteGroundQ2d"
# Region padding tuple in +X,-X,+Y,-Y order (um).
region_padding_um = (100.0, 100.0, 1000.0, 100.0)
# Matrix solve frequency (GHz).
frequency_ghz = 6.0
# Maximum adaptive passes.
maximum_passes = 3

## Create Simulation Component / Coupon

In [ ]:
# Q2D rectangles below are independently authored; no component is built.

## Initialize AEDT Project / App

In [ ]:
# Only the generated native runner owns an AEDT session.

## Build Native Q2D Geometry and Conductors

In [ ]:
if WORKFLOW_ACTION != "analyze_handoff":
    signal_half_width_um = signal_width_um / 2
    left_ground_xmin_um = -(signal_half_width_um + gap_um + ground_width_um)
    right_ground_xmin_um = signal_half_width_um + gap_um
    rectangles = (
        # Each row gives name, lower-left coordinate (um), size (um), and material.
        Q2dRectangleSpec(
            "Substrate",
            (-substrate_width_um / 2, -substrate_thickness_um),
            (substrate_width_um, substrate_thickness_um),
            "Si",
        ),
        Q2dRectangleSpec(
            "Signal",
            (-signal_half_width_um, 0.0),
            (signal_width_um, metal_thickness_um),
            "Nb",
        ),
        Q2dRectangleSpec(
            "GroundLeft",
            (left_ground_xmin_um, 0.0),
            (ground_width_um, metal_thickness_um),
            "Nb",
        ),
        Q2dRectangleSpec(
            "GroundRight",
            (right_ground_xmin_um, 0.0),
            (ground_width_um, metal_thickness_um),
            "Nb",
        ),
    )

## Geometry Verification

In [ ]:
if WORKFLOW_ACTION != "analyze_handoff":
    display(rectangles)

## Materials and Boundaries

In [ ]:
if WORKFLOW_ACTION != "analyze_handoff":
    material_records = get_material_records()
    materials = {
        material_id: PdkMaterial(
            material_id,
            material_records[material_id]["material_kind"],
            material_records[material_id]["is_superconducting"],
            material_records[material_id]["aedt_library_name"],
        )
        for material_id in ("vacuum", "Si", "Nb")
    }

## Ports / Nets / Excitations

In [ ]:
if WORKFLOW_ACTION != "analyze_handoff":
    conductors = (
        # Each row gives conductor name, net name, rectangle members, and thickness (um).
        Q2dConductorSpec("Signal", "SignalLine", ("Signal",), metal_thickness_um),
        Q2dConductorSpec(
            "Ground", "ReferenceGround", ("GroundLeft", "GroundRight"), metal_thickness_um
        ),
    )

## Simulation Setup

In [ ]:
if WORKFLOW_ACTION != "analyze_handoff":
    spec = Q2dSpec(
        project_name=project_name,
        design_name=design_name,
        materials=materials,
        vacuum_material_id="vacuum",
        rectangles=rectangles,
        conductors=conductors,
        run_control=MatrixRunControl("Setup1", frequency_ghz, maximum_passes),
        region_padding_um=region_padding_um,
        aedt_version=aedt_version,
    )

## Simulation Configuration

In [ ]:
if WORKFLOW_ACTION != "analyze_handoff":
    display(spec)

## Solve and Export

In [ ]:
if WORKFLOW_ACTION in {"prepare_handoff", "run"}:
    HANDOFF = prepare_handoff(spec=spec, output_dir=RUN_DIR, resources=RESOURCES)
    display(HANDOFF)
    if WORKFLOW_ACTION == "run":
        subprocess.run([str(HANDOFF.script_path)], cwd=HANDOFF.run_dir, check=True)

## Adaptive-Pass Convergence / Solver Diagnostics

In [ ]:
if WORKFLOW_ACTION in {"run", "analyze_handoff"}:
    result = resolve_results(
        HANDOFF.run_dir if WORKFLOW_ACTION == "run" else Path(RETURNED_RUN_DIR)
    )
    result.show_all_results(show_details=True)

## Results: Plots and Readable Tables

### Physics Analysis Results

In [ ]:
if WORKFLOW_ACTION in {"run", "analyze_handoff"}:
    display(result.physics_results())

### Simulation Performance / Benchmarks

In [ ]:
if WORKFLOW_ACTION in {"run", "analyze_handoff"}:
    display(result.simulation_benchmark())

## Save and Release AEDT

In [ ]:
if WORKFLOW_ACTION in {"run", "analyze_handoff"}:
    display({"project": result.project_path, "receipt": result.receipt_path})
elif WORKFLOW_ACTION in {"prepare_handoff", "run"}:
    display(HANDOFF.archive_path)